In [4]:
!pip install pdfminer.six

In [5]:
import pdfminer
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextBox
import pandas as pd
import numpy as np

In [6]:
import os

pdf_files = [f for f in os.listdir('.') if f.endswith('.pdf')]

if pdf_files:
    pdf_path = pdf_files[0]
    print(f"Используется PDF-файл: {pdf_path}")
else:
    raise Exception("PDF-файлы не найдены в текущем каталоге")

Используется PDF-файл: Publicly Available Clinical BERT Embeddings.pdf


In [7]:
class PDF_to_text_extractor():
  def __init__(self, file_path:str|None = None):
    self.pdf_path = None
    self.text_content = None
    self.boxes = []
    if file_path:
      self.pdf_path = file_path
      # self.text_content = extract_pages(file_path)
    else: print("Файл отсутствует или указан неверный путь.")

  def extract_text_from_boxes(self) -> None:
    try:
        for page_layout in extract_pages(self.pdf_path):
            for element in page_layout:
                if isinstance(element, LTTextBox):
                    # print(f"--- Box at {element.bbox} ---")
                    self.boxes.append([element.bbox, element.get_text().strip()])
    except Exception as e:
        print(f"Ошибка при обработке файла: {e}")

  def _get_text_rects(self):
    pass

In [8]:
document = PDF_to_text_extractor(pdf_path)
document.extract_text_from_boxes()

In [9]:
def collect_pdf_data(doc:PDF_to_text_extractor):
    data = []
    for page_layout in extract_pages(pdf_path):
        page_width = page_layout.width
        page_height = page_layout.height
        for element in page_layout:
            if isinstance(element, LTTextBox):
                text = element.get_text().strip()
                if not text: continue

                # Извлекаем признаки (features)
                x0, y0, x1, y1 = element.bbox
                data.append({
                    'text': text,
                    'len': len(text),
                    'x0': x0 / page_width,  # Нормализация
                    'y0': y0 / page_height,
                    'x1': x1 / page_width,
                    'y1': y1 / page_height,
                    'num_lines': text.count('\n') + 1,
                    'is_article': 1 # Временная метка для обучения
                })
    return pd.DataFrame(data)

df_features = collect_pdf_data(pdf_path)
display(df_features.head())

,text,len,x0,y0,x1,y1,num_lines,is_article
0,Publicly Available Clinical BERT Embeddings,43,0.263328,0.899403,0.740486,0.916444,1,1
1,"Emily Alsentzer\nHarvard-MIT\nCambridge, MA\ne...",56,0.118921,0.815253,0.287622,0.878275,4,1
2,"John R. Murphy\nMIT CSAIL\nCambridge, MA\njrmu...",55,0.305124,0.815253,0.497925,0.878275,4,1
3,"Willie Boag\nMIT CSAIL\nCambridge, MA\nwboag@m...",49,0.516817,0.815253,0.673468,0.878275,4,1
4,"Wei-Hung Weng\nMIT CSAIL\nCambridge, MA\nckbji...",54,0.694742,0.815253,0.887543,0.878275,4,1


In [10]:
# # Автоматическая разметка блоков на основе эвристик для быстрого улучшения
# for i in range(len(df_features)):
#     row = df_features.iloc[i]
#     # Эвристика: короткий текст или текст в самом низу/верху часто является оформлением
#     if row['len'] < 50 or row['y0'] < 0.2 or row['y1'] > 0.8:
#         df_features.at[i, 'is_article'] = 0
#     else:
#         df_features.at[i, 'is_article'] = 1

# # Подготовка данных
# X_updated_heur = df_features[:][features].values
# y_updated_heur = df_features[:]['is_article'].values

# print(f"Обновлено меток: {len(y_updated_heur)}")
# print(f"Распределение классов: {np.bincount(y_updated_heur.astype(int))}")

In [11]:
import tensorflow as tf
from tensorflow.keras import layers

def build_classifier_model(input_shape):
    model = tf.keras.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid') # Бинарная классификация
    ])

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

'''Метки: 0 - оформление, 1 - текст статьи'''

'Метки: 0 - оформление, 1 - текст статьи'

In [12]:
def create_model() -> tf.keras.Sequential:
  # чтение обучающих данных
  training_data = pd.read_csv(TRAIN_DATA_FILENAME)

  # Подготовка признаков (X) и целевой переменной (y)
  features = ['len', 'x0', 'y0', 'x1', 'y1', 'num_lines']
  X_train = training_data[features].values
  y_train = training_data['is_article'].values
  print(f"Загружено строк для обучения: {len(training_data)}")

  # Cоздание и обучение модели
  input_dim = X_train.shape[1]
  model = build_classifier_model(input_dim)

  print("Начало обучения на загруженном датасете...")
  history = model.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)
  print(f"Обучение завершено. Финальная точность: {history.history['accuracy'][-1]:.4f}")
  return model

In [13]:
MODEL_FILENAME = "text_classifier_model.h5"
TRAIN_DATA_FILENAME = "compiled.csv"


model = None
models_list = [f for f in os.listdir('.') if f.endswith('.h5')]
if models_list and (MODEL_FILENAME in models_list):
  print("Найдена предобученная модель")
  model = tf.keras.models.load_model(MODEL_FILENAME)
else:
  print(f"Предобученная модель не найдена. Программа создаст её с нуля на основе обучающих данных из {TRAIN_DATA_FILENAME}")
  model = create_model()
  model.save(MODEL_FILENAME)
  print(f"Модель сохранена в файл {MODEL_FILENAME} для будущего использования")

Предобученная модель не найдена. Программа создаст её с нуля на основе обучающих данных из compiled.csv
Загружено строк для обучения: 798
Начало обучения на загруженном датасете...


Обучение завершено. Финальная точность: 0.8584
Модель сохранена в файл text_classifier_model.h5 для будущего использования


In [14]:
# Предсказание вероятностей
X = df_features[['len', 'x0', 'y0', 'x1', 'y1', 'num_lines']].values
predictions = model.predict(X)

threshold = 0.5
df_features['is_article_prob'] = predictions
df_features['predicted_class'] = (predictions > threshold).astype(int)

# Вывод только тех блоков, которые классифицированы как текст статьи
article_blocks = df_features[df_features['predicted_class'] == 1]
print(f"Найдено блоков текста статьи: {len(article_blocks)} из {len(df_features)}")
display(article_blocks[['text', 'is_article_prob']].head(10))

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
Найдено блоков текста статьи: 29 из 173


,text,is_article_prob
9,Contextual word embedding models such as\nELMo...,0.977025
13,These models have been primarily explored for\...,0.677668
18,2. We demonstrate that using clinical speciﬁc\...,0.733770
28,"2017), express all possible meanings of a word...",0.960361
30,Contextual Clinical & Biomedical Embeddings\nS...,0.799337
31,"On clinical text, (Khin et al., 2018) uses a\n...",0.639704
37,"(Si et al., 2019), released in late February 2...",0.898591
41,We use clinical text from the approximately 2 ...,0.773005
42,We train two varieties of BERT on MIMIC\nnotes...,0.509036
43,Note that we train our clinical BERT instantia...,0.784872


In [15]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(10, 6))
# sns.scatterplot(data=df_features, x='y0', y='is_article_prob', alpha=0.6)
# plt.axhline(y=0.5, color='r', linestyle='--', label=f'Threshold {threshold}')
# plt.title('Зависимость вероятности (is_article) от координаты Y0')
# plt.xlabel('Координата y0 (нормализованная, 0 - низ, 1 - верх)')
# plt.ylabel('Вероятность (Модель)')
# plt.grid(True, linestyle=':', alpha=0.7)
# plt.legend()
# plt.show()

In [16]:
full_text = "\n\n".join(article_blocks['text'].tolist())

file_name = "article_content.txt"
with open(file_name, "w", encoding="utf-8") as f:
    f.write(full_text)

print(f"Текст сохранен в файл: {file_name}")
print(f"Общая длина текста: {len(full_text)} символов.")

Текст сохранен в файл: article_content.txt
Общая длина текста: 15044 символов.


In [17]:
df_features.to_csv('training_dataset.csv', index=False)
print("Датасет сохранен в 'training_dataset.csv'")

Датасет сохранен в 'training_dataset.csv'


In [19]:
def get_sorted_text_from_pdf_multiclmn(pdf_path):
  '''
    Сортировка блоков:
    1. Сначала по Y-координате (сверху вниз, так как y0 для верхнего края)
    2. Затем по X-координате (слева направо, для обработки колонок)
    y0 в pdfminer - это нижняя граница. Для чтения сверху вниз нужно сортировать по y0 в убывающем порядке

    Эвристика для колонок: если есть значительный разрыв по X, это могут быть колонки.
    Надёжней всего сортировать сначала по X, а потом по Y для каждой колонки (но в большинстве случаев достаточно просто сортировать по y0 убывающе, а потом по x0 возрастающе).

    TODO: кластеризация по X-координате для определения колонок (более надёжный способ)
    '''
  full_text = []
  for page_layout in extract_pages(pdf_path):
      page_texts = []
      for element in page_layout:
          if isinstance(element, LTTextBox):
              text = element.get_text().strip()
              if text:
                  x0, y0, x1, y1 = element.bbox
                  page_texts.append({'text': text, 'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1})

      page_texts_sorted = sorted(page_texts, key=lambda b: (-b['y0'], b['x0']))

      # Собираем текст в одну строку для страницы
      full_text.extend([block['text'] for block in page_texts_sorted])

  return "\n\n".join(full_text)